# EXP 02 — Leakage-Safe Historical Strength Engine, Frozen vs Recursive Future Simulation, dan CatBoost Decomposition + Joint Decoder

---

## Ringkasan Eksperimen

Ini adalah **lanjutan langsung dari EXP 01** yang sudah membuktikan bahwa:
- Baseline shared-feature CatBoost valid dan mengalahkan naïve 1-1
- Decomposition + decoder lebih cocok ke AW-MAE (3.108 vs 3.210 vs 4.662 naïve)
- Bottleneck utama: belum ada **dynamic strength representation**

### Apa yang baru di EXP 02?

| Komponen | EXP 01 | EXP 02 |
|---|---|---|
| **Fitur** | Static shared only (44) | Static + Historical strength (107) |
| **Strength Engine** | Tidak ada | Elo + GD rating + EWMA + rolling |
| **H2H** | Tidak ada | Lightweight H2H last-3 |
| **Inferensi Masa Depan** | Batch predict | Frozen vs Recursive simulation |
| **Model** | CatBoost | CatBoost (sama, fair comparison) |
| **Decoder** | Joint integer decoder | Joint integer decoder (sama) |

### Tiga Variant yang Dibandingkan

1. **Control** — reproduce EXP 01 core (static features only)
2. **Frozen** — static + history, tapi performance state dibekukan dari cutoff
3. **Recursive** — static + history, state di-update dengan prediksi model sendiri

### Hipotesis
1. Historical features akan mengalahkan EXP 01 baseline
2. Strength features lebih informatif daripada identity mentah
3. Recursive update lebih realistis untuk horizon panjang
4. Pemisahan state per gender membantu
5. Kombinasi static + history + decoder akan paling kuat

## 01. Setup, Seed, dan Konfigurasi Path

In [1]:
import os
import json
import math
import copy
import random
import warnings
from pathlib import Path
from itertools import product
from collections import Counter, deque

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from catboost import CatBoostRegressor, CatBoostClassifier, Pool

print("[INFO] Semua library berhasil di-import.")

[INFO] Semua library berhasil di-import.


In [2]:
# === KONFIGURASI GLOBAL ===
SEED = 42

TRAIN_PATH = "../data/train.csv"
TEST_PATH  = "../data/test.csv"
SAMPLE_SUB_PATH = "../data/sample submission.csv"
META_PATH  = "../data/metadata.txt"

OUT_ROOT = "../outputs/exp02_history_strength_engine_recursive"
FIG_DIR  = f"{OUT_ROOT}/figures"
PRED_DIR = f"{OUT_ROOT}/predictions"
SUB_DIR  = f"{OUT_ROOT}/submissions"
SUM_DIR  = f"{OUT_ROOT}/summaries"

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
warnings.filterwarnings("ignore")

print(f"[INFO] SEED = {SEED}")

[INFO] SEED = 42


In [3]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[INFO] Seed set to {seed}")

seed_everything(SEED)

[INFO] Seed set to 42


In [4]:
# Buat folder output
for d in [FIG_DIR, PRED_DIR, SUB_DIR, SUM_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"[OK] {d}")

[OK] ../outputs/exp02_history_strength_engine_recursive/figures
[OK] ../outputs/exp02_history_strength_engine_recursive/predictions
[OK] ../outputs/exp02_history_strength_engine_recursive/submissions
[OK] ../outputs/exp02_history_strength_engine_recursive/summaries


### Catatan Setup
- Notebook ini **standalone** — tidak bergantung pada notebook EXP 00 atau EXP 01.
- Semua helper penting ditulis ulang di notebook ini.
- Folder output dibuat eksplisit untuk menyimpan artifact eksperimen.

## 02. Validasi File Input

In [5]:
FILE_PATHS = {
    "train": TRAIN_PATH,
    "test": TEST_PATH,
    "sample_submission": SAMPLE_SUB_PATH,
    "metadata": META_PATH,
}

def validate_input_files(file_paths: dict) -> None:
    missing = []
    for name, path in file_paths.items():
        if not Path(path).exists():
            missing.append((name, path))
    if missing:
        msg = "File TIDAK ditemukan:\n" + "\n".join(
            f"  - {n}: {p}" for n, p in missing
        )
        raise FileNotFoundError(msg)
    print("[OK] Semua file input ditemukan.")

validate_input_files(FILE_PATHS)

[OK] Semua file input ditemukan.


## 03. Load Data dan Helper Inti (standalone dari EXP 00/01)

In [6]:
# --- Load data ---
train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
test  = pd.read_csv(TEST_PATH,  parse_dates=["date"])
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Sample: {sample_sub.shape}")

Train : (78772, 47)
Test  : (42422, 20)
Sample: (42422, 3)


In [7]:
# ============================================================
# HELPER FUNCTIONS  (standalone, ditulis ulang dari EXP 00/01)
# ============================================================

def build_match_level(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    """Konversi row-level -> match-level. Canonical: team_a = alfabet pertama."""
    counts = df.groupby("match_id").size()
    bad = counts[counts != 2]
    if len(bad) > 0:
        raise ValueError(f"{len(bad)} match_id tanpa tepat 2 rows")

    df_sorted = df.sort_values(["match_id", "team"]).reset_index(drop=True)
    row_a = df_sorted.iloc[0::2].reset_index(drop=True)
    row_b = df_sorted.iloc[1::2].reset_index(drop=True)
    assert (row_a["match_id"].values == row_b["match_id"].values).all()

    result = pd.DataFrame()
    for col in ["match_id", "date", "gender", "tournament", "venue_country", "neutral"]:
        if col in df.columns:
            result[col] = row_a[col].values
    for col in ["altitude_venue", "temperature_venue"]:
        if col in df.columns:
            result[col] = row_a[col].values

    result["team_a"] = row_a["team"].values
    result["team_b"] = row_b["team"].values
    if "is_home" in df.columns:
        result["team_a_is_home"] = row_a["is_home"].values
        result["team_b_is_home"] = row_b["is_home"].values

    sym = {"confederation_team":"confederation","population_team":"population",
           "gdp_per_capita_team":"gdp_per_capita","distance_travel_team":"distance_travel"}
    for orig, suf in sym.items():
        if orig in df.columns:
            result[f"team_a_{suf}"] = row_a[orig].values
            result[f"team_b_{suf}"] = row_b[orig].values

    if is_train and "team_goals" in df.columns:
        result["team_a_goals"] = row_a["team_goals"].values
        result["team_b_goals"] = row_a["opp_goals"].values

    return result


def match_predictions_to_submission(
    test_row_df, pred_match_df,
    canonical_team_a_col="team_a", canonical_team_b_col="team_b",
    pred_a_col="pred_team_a_goals", pred_b_col="pred_team_b_goals",
):
    """Konversi prediksi match-level -> submission row-level."""
    merge_cols = ["match_id", canonical_team_a_col, canonical_team_b_col,
                  pred_a_col, pred_b_col]
    merged = test_row_df[["Id","match_id","team"]].merge(
        pred_match_df[merge_cols], on="match_id", how="left",
    )
    is_a = merged["team"] == merged[canonical_team_a_col]
    merged["team_goals"] = np.where(is_a, merged[pred_a_col], merged[pred_b_col])
    merged["opp_goals"]  = np.where(is_a, merged[pred_b_col], merged[pred_a_col])
    return merged[["Id","team_goals","opp_goals"]].copy()


# --- AW-MAE Evaluator ---
EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50

def _outcome(a, b):
    a, b = float(a), float(b)
    return 1 if a > b else (0 if a == b else -1)

def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()
    if "fifa world cup" in t or t == "world cup":
        return 2.00
    elif "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80
    elif "friendly" in t:
        return 0.96
    else:
        return 1.20

def official_match_loss(y_team_true, y_opp_true, y_team_pred, y_opp_pred):
    yt, yo = float(y_team_true), float(y_opp_true)
    pt, po = float(y_team_pred), float(y_opp_pred)
    mae = (abs(yt - pt) + abs(yo - po)) / 2.0
    exact = 1 if (yt == pt and yo == po) else 0
    oc = 1 if _outcome(yt, yo) == _outcome(pt, po) else 0
    gdc = 1 if (yt - yo) == (pt - po) else 0
    penalty = EXACT_PENALTY*(1-exact) + OUTCOME_PENALTY*(1-oc) + GD_PENALTY*(1-gdc)
    mult = 1.0 if oc else WRONG_OUTCOME_MULTIPLIER
    return ((mae + penalty) * mult) ** NONLINEAR_POWER

def awmae_score(y_team_true, y_opp_true, y_team_pred, y_opp_pred, tournaments):
    """Loop-based AW-MAE reference."""
    yt = np.asarray(y_team_true, dtype=float)
    yo = np.asarray(y_opp_true, dtype=float)
    pt = np.asarray(y_team_pred, dtype=float)
    po = np.asarray(y_opp_pred, dtype=float)
    n = len(yt)
    tlist = tournaments.tolist() if hasattr(tournaments, "tolist") else list(tournaments)
    wl, wt = 0.0, 0.0
    for i in range(n):
        loss = official_match_loss(yt[i], yo[i], pt[i], po[i])
        w = get_tournament_weight(str(tlist[i]))
        wl += loss * w
        wt += w
    return wl / wt if wt > 0 else 0.0

def awmae_score_fast(y_a_true, y_b_true, y_a_pred, y_b_pred, weights):
    """Vectorized AW-MAE for efficient grid search."""
    yt_a = np.asarray(y_a_true, dtype=float)
    yt_b = np.asarray(y_b_true, dtype=float)
    yp_a = np.asarray(y_a_pred, dtype=float)
    yp_b = np.asarray(y_b_pred, dtype=float)
    w = np.asarray(weights, dtype=float)
    mae = (np.abs(yt_a - yp_a) + np.abs(yt_b - yp_b)) / 2.0
    exact = ((yt_a == yp_a) & (yt_b == yp_b)).astype(float)
    sign_t = np.sign(yt_a - yt_b)
    sign_p = np.sign(yp_a - yp_b)
    oc = (sign_t == sign_p).astype(float)
    gdc = ((yt_a - yt_b) == (yp_a - yp_b)).astype(float)
    penalty = 0.30*(1-exact) + 0.25*(1-oc) + 0.15*(1-gdc)
    mult = np.where(oc, 1.0, 1.5)
    loss = ((mae + penalty) * mult) ** 1.5
    return np.sum(loss * w) / np.sum(w) if np.sum(w) > 0 else 0.0

def make_time_based_holdout(train_match, valid_fraction=0.2):
    """Split temporal: 20% match terakhir -> validation."""
    df = train_match.sort_values("date").reset_index(drop=True)
    n = len(df)
    idx = int(n * (1 - valid_fraction))
    tr = df.iloc[:idx].reset_index(drop=True)
    vl = df.iloc[idx:].reset_index(drop=True)
    print(f"Train fold: {len(tr):,} matches  ({tr['date'].min()} -> {tr['date'].max()})")
    print(f"Valid fold: {len(vl):,} matches  ({vl['date'].min()} -> {vl['date'].max()})")
    return tr, vl

def build_outcome_target(goal_a, goal_b):
    """Outcome class: 0=team_a_win, 1=draw, 2=team_b_win."""
    return np.where(goal_a > goal_b, 0, np.where(goal_a == goal_b, 1, 2))

def build_scoreline_prior(goal_a, goal_b, max_goals, alpha=1.0):
    """Smoothed probability distribution atas grid scoreline."""
    counts = Counter(zip(
        np.clip(np.asarray(goal_a, dtype=int), 0, max_goals),
        np.clip(np.asarray(goal_b, dtype=int), 0, max_goals),
    ))
    total = sum(counts.values()) + alpha * (max_goals + 1) ** 2
    prior = {}
    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            prior[(a, b)] = (counts.get((a, b), 0) + alpha) / total
    return prior

def decode_single_match_score(
    pred_goal_a, pred_goal_b, pred_total, pred_gd, pred_outcome_proba,
    scoreline_prior, max_goals,
    w_direct=1.0, w_total=1.0, w_gd=1.0, w_outcome=1.0, w_prior=0.2,
    eps=1e-9,
):
    """Decode skor satu pertandingan."""
    best_cost, best_a, best_b = float("inf"), 0, 0
    for a in range(max_goals + 1):
        for b in range(max_goals + 1):
            cost = w_direct * (abs(a - pred_goal_a) + abs(b - pred_goal_b))
            cost += w_total * abs((a + b) - pred_total)
            cost += w_gd * abs((a - b) - pred_gd)
            oi = 0 if a > b else (1 if a == b else 2)
            cost += w_outcome * (-math.log(pred_outcome_proba[oi] + eps))
            pp = scoreline_prior.get((a, b), eps)
            cost += w_prior * (-math.log(pp + eps))
            if cost < best_cost:
                best_cost, best_a, best_b = cost, a, b
    return best_a, best_b

def prepare_features(df, feat_list, cat_feats):
    """Siapkan DataFrame fitur untuk CatBoost."""
    X = df[feat_list].copy()
    for col in cat_feats:
        if col in X.columns:
            X[col] = X[col].fillna("MISSING").astype(str)
    return X

print("[OK] Semua helper berhasil didefinisikan.")

[OK] Semua helper berhasil didefinisikan.


### Catatan Helper
Seluruh helper inti ditulis ulang agar notebook **standalone**.
Ditambahkan `awmae_score_fast` (vectorized) untuk grid search efisien.

## 04. Cleaning Awal dan Canonical Base Match Data

Cleaning konservatif:
- Sentinel `altitude_venue == -9999` → `NaN`
- Kolom kategorikal bertipe string
- Canonicalization identik dengan EXP 00/01

In [8]:
# Bersihkan sentinel altitude_venue == -9999
SENTINEL = -9999
for name, df in [("train", train), ("test", test)]:
    if "altitude_venue" in df.columns:
        n_sent = (df["altitude_venue"] == SENTINEL).sum()
        df.loc[df["altitude_venue"] == SENTINEL, "altitude_venue"] = np.nan
        print(f"  {name}: {n_sent:,} sentinel altitude_venue -> NaN")

# Pastikan kategorikal string
CAT_RAW_COLS = ["team","opponent","tournament","venue_country","gender",
                "confederation_team","confederation_opp"]
for col in CAT_RAW_COLS:
    for df in [train, test]:
        if col in df.columns:
            df[col] = df[col].astype(str)
print("[OK] Kolom kategorikal aman sebagai string.")

  train: 766 sentinel altitude_venue -> NaN
  test: 296 sentinel altitude_venue -> NaN
[OK] Kolom kategorikal aman sebagai string.


In [9]:
# Bangun match-level canonical
train_match_base = build_match_level(train, is_train=True)
test_match_base  = build_match_level(test,  is_train=False)

print(f"train_match_base: {train_match_base.shape}")
print(f"test_match_base : {test_match_base.shape}")
assert len(train_match_base) == train["match_id"].nunique()
assert len(test_match_base)  == test["match_id"].nunique()
print("[OK] Match-level canonical berhasil dibangun.")
display(train_match_base.head(3))

train_match_base: (39386, 22)
test_match_base : (21211, 20)
[OK] Match-level canonical berhasil dibangun.


,match_id,date,gender,tournament,venue_country,neutral,altitude_venue,temperature_venue,team_a,team_b,team_a_is_home,team_b_is_home,team_a_confederation,team_b_confederation,team_a_population,team_b_population,team_a_gdp_per_capita,team_b_gdp_per_capita,team_a_distance_travel,team_b_distance_travel,team_a_goals,team_b_goals
0,M000001,1872-11-30,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,M000002,1873-03-08,M,Friendly,England,0,NaN,NaN,England,Scotland,1,0,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,4,2
2,M000003,1874-03-07,M,Friendly,Scotland,0,NaN,NaN,England,Scotland,0,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,1,2


Canonicalization identik dengan EXP 00/01 (urut alfabet) agar perbandingan antar eksperimen tetap adil.

## 05. Static Shared Features (Control Block dari EXP 01)

Fitur static shared yang sama persis dengan EXP 01 sebagai **control feature block**.

In [10]:
def engineer_static_match_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tambahkan fitur static turunan ke DataFrame match-level.
    Identik dengan EXP 01, bekerja untuk train_match dan test_match."""
    df = df.copy()

    # --- 1. Date features ---
    dt = pd.to_datetime(df["date"])
    df["match_year"]      = dt.dt.year
    df["match_month"]     = dt.dt.month
    df["match_quarter"]   = dt.dt.quarter
    df["match_dayofweek"] = dt.dt.dayofweek
    df["match_dayofyear"] = dt.dt.dayofyear
    df["match_is_weekend"]= (dt.dt.dayofweek >= 5).astype(int)
    df["match_decade"]    = (dt.dt.year // 10) * 10

    # --- 2. Home / venue / confederation ---
    df["home_side"] = np.where(
        df["team_a_is_home"] == 1, 1,
        np.where(df["team_b_is_home"] == 1, -1, 0)
    )
    conf_a = df["team_a_confederation"].fillna("UNKNOWN").astype(str)
    conf_b = df["team_b_confederation"].fillna("UNKNOWN").astype(str)
    df["same_confederation"] = (conf_a == conf_b).astype(int)

    t_low = df["tournament"].str.lower().fillna("")
    df["is_friendly"]       = t_low.str.contains("friendly").astype(int)
    df["is_world_cup"]      = t_low.str.contains("world cup").astype(int)
    df["is_qualification"]  = t_low.str.contains("qualif").astype(int)
    df["is_nations_league"] = t_low.str.contains("nations league").astype(int)
    df["tournament_weight_proxy"] = df["tournament"].apply(
        lambda x: get_tournament_weight(str(x))
    )

    # --- 3. Numeric symmetry features ---
    pairs = [
        ("population", "team_a_population", "team_b_population"),
        ("gdp",        "team_a_gdp_per_capita", "team_b_gdp_per_capita"),
        ("distance",   "team_a_distance_travel", "team_b_distance_travel"),
    ]
    for prefix, col_a, col_b in pairs:
        a = df[col_a].astype(float)
        b = df[col_b].astype(float)
        df[f"{prefix}_diff"]     = a - b
        df[f"{prefix}_abs_diff"] = np.abs(a - b)
        df[f"log_{prefix}_a"]    = np.log1p(np.abs(a))
        df[f"log_{prefix}_b"]    = np.log1p(np.abs(b))
        df[f"log_{prefix}_diff"] = np.log1p(np.abs(a)) - np.log1p(np.abs(b))
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where((b == 0) | b.isna(), np.nan, a / b)
        ratio = np.where(np.isinf(ratio), np.nan, ratio)
        df[f"{prefix}_ratio_ab"] = ratio

    # --- 4. Matchup category features ---
    df["pair_key"] = df["team_a"].astype(str) + "__VS__" + df["team_b"].astype(str)
    df["confed_pair_key"] = (
        df["team_a_confederation"].fillna("UNK").astype(str)
        + "__VS__"
        + df["team_b_confederation"].fillna("UNK").astype(str)
    )
    return df


# Terapkan
train_static = engineer_static_match_features(train_match_base)
test_static  = engineer_static_match_features(test_match_base)

print(f"train_static: {train_static.shape}")
print(f"test_static : {test_static.shape}")

train_static: (39386, 56)
test_static : (21211, 54)


Static features identik dengan EXP 01 untuk menjamin perbandingan fair antar variant.

## 06. Historical Strength Engine — Definisi State dan Helper

### Konsep Engine
Engine ini membangun **pre-match features** dari histori pertandingan sebelumnya.

Setiap tim memiliki **state** unik per `(gender, team)`:
- Elo rating (outcome + goal-diff)
- EWMA form (points, goals for/against, goal diff)
- Rolling recent performance (last 5/10 matches)
- Rest days (from last match date)

Setiap pasangan tim memiliki **H2H state** per `(gender, team_a, team_b)`:
- Recent H2H points, goal diff, total goals

In [11]:
# ============================================================
# TEAM STATE MANAGEMENT
# ============================================================

def init_team_state() -> dict:
    """Inisialisasi state default untuk tim baru."""
    return {
        "matches_played": 0,
        "last_match_date": None,
        # Elo ratings
        "elo_overall": 1500.0,
        "elo_goal_diff": 0.0,
        # EWMA form
        "ewm_points": 1.0,
        "ewm_goals_for": 1.2,
        "ewm_goals_against": 1.2,
        "ewm_goal_diff": 0.0,
        # Rolling deques
        "recent_points_5": deque(maxlen=5),
        "recent_points_10": deque(maxlen=10),
        "recent_gf_5": deque(maxlen=5),
        "recent_ga_5": deque(maxlen=5),
        "recent_gd_5": deque(maxlen=5),
        "recent_results_10": deque(maxlen=10),  # 1.0=win, 0.5=draw, 0.0=loss
        "recent_clean_sheet_5": deque(maxlen=5),
        "recent_failed_to_score_5": deque(maxlen=5),
    }


def init_h2h_state() -> dict:
    """Inisialisasi H2H state untuk pasangan tim baru."""
    return {
        "h2h_matches_played": 0,
        "h2h_points_a_last3": deque(maxlen=3),
        "h2h_gd_a_last3": deque(maxlen=3),
        "h2h_total_goals_last3": deque(maxlen=3),
    }


def _safe_deque_mean(d, default=0.0):
    """Mean of deque, fallback ke default jika kosong."""
    return sum(d) / len(d) if len(d) > 0 else default


def expected_elo_result(rating_a: float, rating_b: float, home_bonus: float = 0.0) -> float:
    """Expected result sisi A menggunakan rumus Elo standar."""
    return 1.0 / (1.0 + 10.0 ** (-(rating_a + home_bonus - rating_b) / 400.0))


print("[OK] State definitions siap.")

[OK] State definitions siap.


In [12]:
# ============================================================
# STATE UPDATE FUNCTIONS
# ============================================================

K_ELO = 24.0
K_GD  = 6.0
ELO_HOME_BONUS = 60.0
GD_HOME_BONUS  = 10.0
EWMA_ALPHA = 0.35

def update_states_from_score(
    state_a: dict, state_b: dict, h2h_state: dict,
    match_context: dict, goals_a: int, goals_b: int,
) -> None:
    """Update team + H2H states in-place given a match result.
    goals_a/goals_b = integer scores (actual or predicted)."""
    goals_a, goals_b = int(goals_a), int(goals_b)
    goal_diff = goals_a - goals_b

    # --- Points & result ---
    if goals_a > goals_b:
        points_a, points_b, result_a, result_b = 3, 0, 1.0, 0.0
    elif goals_a == goals_b:
        points_a, points_b, result_a, result_b = 1, 1, 0.5, 0.5
    else:
        points_a, points_b, result_a, result_b = 0, 3, 0.0, 1.0

    # --- Home bonus ---
    home_a = float(match_context.get("home_a", 0))
    home_b = float(match_context.get("home_b", 0))
    elo_hb = ELO_HOME_BONUS * home_a - ELO_HOME_BONUS * home_b
    gd_hb  = GD_HOME_BONUS * home_a - GD_HOME_BONUS * home_b

    # --- Elo outcome update ---
    tournament_w = get_tournament_weight(match_context.get("tournament", ""))
    exp_a = expected_elo_result(state_a["elo_overall"], state_b["elo_overall"], elo_hb)
    delta_elo = K_ELO * tournament_w * (result_a - exp_a)
    state_a["elo_overall"] += delta_elo
    state_b["elo_overall"] -= delta_elo

    # --- Goal-diff rating update ---
    exp_gd = (state_a["elo_goal_diff"] + gd_hb - state_b["elo_goal_diff"]) / 100.0
    resid_gd = goal_diff - exp_gd
    state_a["elo_goal_diff"] += K_GD * resid_gd
    state_b["elo_goal_diff"] -= K_GD * resid_gd

    # --- EWMA form update ---
    alpha = EWMA_ALPHA
    state_a["ewm_points"]        = alpha * points_a + (1-alpha) * state_a["ewm_points"]
    state_b["ewm_points"]        = alpha * points_b + (1-alpha) * state_b["ewm_points"]
    state_a["ewm_goals_for"]     = alpha * goals_a  + (1-alpha) * state_a["ewm_goals_for"]
    state_b["ewm_goals_for"]     = alpha * goals_b  + (1-alpha) * state_b["ewm_goals_for"]
    state_a["ewm_goals_against"] = alpha * goals_b  + (1-alpha) * state_a["ewm_goals_against"]
    state_b["ewm_goals_against"] = alpha * goals_a  + (1-alpha) * state_b["ewm_goals_against"]
    state_a["ewm_goal_diff"]     = alpha * goal_diff  + (1-alpha) * state_a["ewm_goal_diff"]
    state_b["ewm_goal_diff"]     = alpha * (-goal_diff) + (1-alpha) * state_b["ewm_goal_diff"]

    # --- Rolling deques ---
    for st, pts, res, gf, ga, gd_val in [
        (state_a, points_a, result_a, goals_a, goals_b,  goal_diff),
        (state_b, points_b, result_b, goals_b, goals_a, -goal_diff),
    ]:
        st["recent_points_5"].append(pts)
        st["recent_points_10"].append(pts)
        st["recent_gf_5"].append(gf)
        st["recent_ga_5"].append(ga)
        st["recent_gd_5"].append(gd_val)
        st["recent_results_10"].append(res)
        st["recent_clean_sheet_5"].append(1 if ga == 0 else 0)
        st["recent_failed_to_score_5"].append(1 if gf == 0 else 0)

    # --- Match count & date ---
    match_date = match_context.get("match_date")
    state_a["matches_played"] += 1
    state_b["matches_played"] += 1
    state_a["last_match_date"] = match_date
    state_b["last_match_date"] = match_date

    # --- H2H update ---
    h2h_state["h2h_matches_played"] += 1
    h2h_state["h2h_points_a_last3"].append(points_a)
    h2h_state["h2h_gd_a_last3"].append(goal_diff)
    h2h_state["h2h_total_goals_last3"].append(goals_a + goals_b)


print("[OK] State update functions siap.")

[OK] State update functions siap.


In [13]:
# ============================================================
# FEATURE SNAPSHOT FUNCTIONS
# ============================================================

def snapshot_team_features(state: dict, match_date) -> dict:
    """Ekstrak pre-match features dari team state."""
    f = {}
    f["hist_matches_played"] = state["matches_played"]
    f["hist_elo_overall"]    = state["elo_overall"]
    f["hist_elo_gd"]         = state["elo_goal_diff"]
    f["hist_ewm_points"]     = state["ewm_points"]
    f["hist_ewm_gf"]         = state["ewm_goals_for"]
    f["hist_ewm_ga"]         = state["ewm_goals_against"]
    f["hist_ewm_gd"]         = state["ewm_goal_diff"]

    f["hist_points_avg_last5"]  = _safe_deque_mean(state["recent_points_5"], 1.0)
    f["hist_points_avg_last10"] = _safe_deque_mean(state["recent_points_10"], 1.0)
    f["hist_gf_avg_last5"]      = _safe_deque_mean(state["recent_gf_5"], 1.2)
    f["hist_ga_avg_last5"]      = _safe_deque_mean(state["recent_ga_5"], 1.2)
    f["hist_gd_avg_last5"]      = _safe_deque_mean(state["recent_gd_5"], 0.0)

    results = list(state["recent_results_10"])
    if len(results) > 0:
        n_r = len(results)
        f["hist_win_rate_last10"]  = sum(1 for r in results if r == 1.0)  / n_r
        f["hist_draw_rate_last10"] = sum(1 for r in results if r == 0.5)  / n_r
        f["hist_loss_rate_last10"] = sum(1 for r in results if r == 0.0)  / n_r
    else:
        f["hist_win_rate_last10"]  = 0.33
        f["hist_draw_rate_last10"] = 0.33
        f["hist_loss_rate_last10"] = 0.33

    f["hist_clean_sheet_rate_last5"]     = _safe_deque_mean(state["recent_clean_sheet_5"], 0.3)
    f["hist_failed_to_score_rate_last5"] = _safe_deque_mean(state["recent_failed_to_score_5"], 0.2)

    # Rest days
    if state["last_match_date"] is not None and pd.notna(match_date):
        delta = (pd.Timestamp(match_date) - pd.Timestamp(state["last_match_date"])).days
        f["hist_days_since_last_match"] = max(delta, 0)
    else:
        f["hist_days_since_last_match"] = np.nan

    f["hist_has_history"] = 1 if state["matches_played"] > 0 else 0
    return f


def snapshot_h2h_features(h2h_state: dict) -> dict:
    """Ekstrak pre-match H2H features."""
    f = {}
    f["h2h_matches_played_pre"]    = h2h_state["h2h_matches_played"]
    f["h2h_points_a_avg_last3"]    = _safe_deque_mean(h2h_state["h2h_points_a_last3"], 1.0)
    f["h2h_gd_a_avg_last3"]        = _safe_deque_mean(h2h_state["h2h_gd_a_last3"], 0.0)
    f["h2h_total_goals_avg_last3"] = _safe_deque_mean(h2h_state["h2h_total_goals_last3"], 2.0)
    f["h2h_has_history"]           = 1 if h2h_state["h2h_matches_played"] > 0 else 0
    return f


def build_history_feature_row(row, team_states: dict, h2h_states: dict) -> dict:
    """Build complete history feature dict for one match row."""
    gender  = str(row.get("gender", "M") if hasattr(row, "get") else row["gender"])
    team_a  = str(row.get("team_a", "")  if hasattr(row, "get") else row["team_a"])
    team_b  = str(row.get("team_b", "")  if hasattr(row, "get") else row["team_b"])
    match_date = row.get("date") if hasattr(row, "get") else row["date"]

    key_a   = (gender, team_a)
    key_b   = (gender, team_b)
    h2h_key = (gender, team_a, team_b)

    sa = team_states.get(key_a, init_team_state())
    sb = team_states.get(key_b, init_team_state())
    sh = h2h_states.get(h2h_key, init_h2h_state())

    fa = snapshot_team_features(sa, match_date)
    fb = snapshot_team_features(sb, match_date)
    fh = snapshot_h2h_features(sh)

    feats = {}
    for k, v in fa.items():
        feats[k + "_a"] = v
    for k, v in fb.items():
        feats[k + "_b"] = v
    feats.update(fh)

    # --- Matchup delta features ---
    delta_bases = [
        "hist_elo_overall", "hist_elo_gd",
        "hist_ewm_points", "hist_ewm_gf", "hist_ewm_ga", "hist_ewm_gd",
        "hist_points_avg_last5", "hist_points_avg_last10",
        "hist_gf_avg_last5", "hist_ga_avg_last5", "hist_gd_avg_last5",
        "hist_win_rate_last10", "hist_draw_rate_last10", "hist_loss_rate_last10",
        "hist_clean_sheet_rate_last5", "hist_failed_to_score_rate_last5",
    ]
    for base in delta_bases:
        va = feats.get(f"{base}_a", 0.0)
        vb = feats.get(f"{base}_b", 0.0)
        feats[f"{base}_diff"] = va - vb

    # Abs diff for Elo features
    for base in ["hist_elo_overall", "hist_elo_gd"]:
        feats[f"{base}_abs_diff"] = abs(feats.get(f"{base}_diff", 0.0))

    # Rest days diff
    rda = feats.get("hist_days_since_last_match_a", np.nan)
    rdb = feats.get("hist_days_since_last_match_b", np.nan)
    if pd.notna(rda) and pd.notna(rdb):
        feats["hist_rest_days_diff"] = rda - rdb
    else:
        feats["hist_rest_days_diff"] = np.nan

    feats["hist_matches_played_diff"] = feats["hist_matches_played_a"] - feats["hist_matches_played_b"]

    return feats


print("[OK] Feature snapshot functions siap.")

[OK] Feature snapshot functions siap.


### Catatan History Engine

- **Team state** dipisahkan per `(gender, team)` karena distribusi men's/women's berbeda.
- **H2H state** per `(gender, team_a, team_b)` canonical.
- `last_match_date` selalu di-track — bahkan dalam mode freeze — karena **rest days** depend on schedule.
- Inisialisasi default stabil: Elo 1500, EWM default rendah, flag `has_history=0`.

## 07. Build Leakage-Safe Historical Features untuk Train

Membangun **pre-match historical features** untuk seluruh match train.
Setiap match hanya melihat histori **sebelum** match tersebut.

In [14]:
def build_train_history_features(train_match_df: pd.DataFrame):
    """Bangun history features chronologis dan leakage-safe.

    Returns
    -------
    hist_df : DataFrame with match_id + history features
    final_team_states : dict (gender, team) -> state
    final_h2h_states  : dict (gender, team_a, team_b) -> state
    """
    df = train_match_df.sort_values("date").reset_index(drop=True)

    team_states = {}
    h2h_states  = {}
    all_rows    = []

    for i in tqdm(range(len(df)), desc="Building history"):
        row = df.iloc[i]
        gender  = str(row["gender"])
        team_a  = str(row["team_a"])
        team_b  = str(row["team_b"])
        match_date = row["date"]
        goals_a = int(row["team_a_goals"])
        goals_b = int(row["team_b_goals"])

        key_a   = (gender, team_a)
        key_b   = (gender, team_b)
        h2h_key = (gender, team_a, team_b)

        # Ensure states exist
        if key_a not in team_states:   team_states[key_a] = init_team_state()
        if key_b not in team_states:   team_states[key_b] = init_team_state()
        if h2h_key not in h2h_states:  h2h_states[h2h_key] = init_h2h_state()

        # 1. Snapshot PRE-MATCH features
        hist_feats = build_history_feature_row(row, team_states, h2h_states)
        hist_feats["match_id"] = row["match_id"]
        all_rows.append(hist_feats)

        # 2. Update states with ACTUAL outcome
        match_ctx = {
            "home_a": row.get("team_a_is_home", 0),
            "home_b": row.get("team_b_is_home", 0),
            "tournament": str(row.get("tournament", "")),
            "match_date": match_date,
        }
        update_states_from_score(
            team_states[key_a], team_states[key_b],
            h2h_states[h2h_key], match_ctx, goals_a, goals_b,
        )

    hist_df = pd.DataFrame(all_rows)
    print(f"  History features: {hist_df.shape[1] - 1} features for {len(hist_df):,} matches")
    return hist_df, team_states, h2h_states


# Bangun history untuk SELURUH train
train_hist_df, final_ts_all, final_hs_all = build_train_history_features(train_match_base)

Building history:   0%|          | 0/39386 [00:00<?, ?it/s]

  History features: 63 features for 39,386 matches


In [15]:
# Gabungkan static + history features
train_full = train_static.merge(train_hist_df, on="match_id", how="left")
print(f"train_full shape: {train_full.shape}")

# Simpan snapshot final state sebagai referensi
state_records = []
for (gender, team), st in final_ts_all.items():
    state_records.append({
        "gender": gender, "team": team,
        "matches_played": st["matches_played"],
        "elo_overall": st["elo_overall"],
        "elo_goal_diff": st["elo_goal_diff"],
        "ewm_points": st["ewm_points"],
    })
pd.DataFrame(state_records).to_csv(f"{SUM_DIR}/cutoff_team_states.csv", index=False)
print(f"[OK] Final state {len(state_records)} teams disimpan.")

train_full shape: (39386, 119)
[OK] Final state 483 teams disimpan.


Semua history features bersifat **pre-match**: snapshot dulu, baru update state. Tidak ada leakage.

## 08. Temporal Holdout dan State Cutoff

In [16]:
# Split train_full (yang sudah punya semua fitur) secara temporal
train_fold, valid_fold = make_time_based_holdout(train_full, valid_fraction=0.2)

# Verifikasi
t_max = train_fold["date"].max()
v_min = valid_fold["date"].min()
print(f"\nTrain fold max date: {t_max}")
print(f"Valid fold min date: {v_min}")
overlap = set(train_fold["match_id"]) & set(valid_fold["match_id"])
assert len(overlap) == 0, f"Overlap! {len(overlap)}"
print("[OK] Tidak ada overlap.")

Train fold: 31,508 matches  (1872-11-30 00:00:00 -> 2005-01-30 00:00:00)
Valid fold: 7,878 matches  (2005-02-01 00:00:00 -> 2011-08-04 00:00:00)

Train fold max date: 2005-01-30 00:00:00
Valid fold min date: 2005-02-01 00:00:00
[OK] Tidak ada overlap.


In [17]:
# Bangun cutoff states khusus train_fold
# (yang hanya melihat actual outcomes dari train_fold, bukan valid_fold)
train_fold_base = train_match_base[train_match_base["match_id"].isin(train_fold["match_id"])].copy()
_, cutoff_ts, cutoff_hs = build_train_history_features(train_fold_base)

print(f"Cutoff team states: {len(cutoff_ts)} entries")
print(f"Cutoff H2H states:  {len(cutoff_hs)} entries")

# Valid fold static (untuk simulasi — tanpa history features)
valid_fold_static = train_static[train_static["match_id"].isin(valid_fold["match_id"])].copy()
valid_fold_static = valid_fold_static.sort_values("date").reset_index(drop=True)
print(f"valid_fold_static: {valid_fold_static.shape}")

Building history:   0%|          | 0/31508 [00:00<?, ?it/s]

  History features: 63 features for 31,508 matches
Cutoff team states: 414 entries
Cutoff H2H states:  5715 entries
valid_fold_static: (7878, 56)


### Catatan Split
- Split dilakukan per **match** (bukan per row), 20% terakhir.
- Cutoff state dibangun hanya dari **train_fold actual history**.
- Validation **tidak boleh** memakai actual valid outcomes untuk state update.

## 09. Target Construction dan Final Feature Set

In [18]:
# --- Targets ---
y_ga_train = train_fold["team_a_goals"].values.astype(float)
y_gb_train = train_fold["team_b_goals"].values.astype(float)
y_total_train   = (y_ga_train + y_gb_train)
y_gd_train      = (y_ga_train - y_gb_train)
y_outcome_train = build_outcome_target(y_ga_train, y_gb_train).astype(int)

y_ga_valid = valid_fold["team_a_goals"].values.astype(float)
y_gb_valid = valid_fold["team_b_goals"].values.astype(float)
y_total_valid   = (y_ga_valid + y_gb_valid)
y_gd_valid      = (y_ga_valid - y_gb_valid)
y_outcome_valid = build_outcome_target(y_ga_valid, y_gb_valid).astype(int)

print("Distribusi outcome (train fold):")
for cls, name in [(0,"A_win"),(1,"Draw"),(2,"B_win")]:
    n = (y_outcome_train == cls).sum()
    print(f"  {cls} ({name}): {n:,} ({n/len(y_outcome_train)*100:.1f}%)")

Distribusi outcome (train fold):
  0 (A_win): 12,297 (39.0%)
  1 (Draw): 6,846 (21.7%)
  2 (B_win): 12,365 (39.2%)


In [19]:
# --- Feature Set Definitions ---
static_categorical = [
    "team_a", "team_b", "gender", "tournament", "venue_country",
    "team_a_confederation", "team_b_confederation",
    "pair_key", "confed_pair_key",
]

static_numeric = [
    "match_year", "match_month", "match_quarter",
    "match_dayofweek", "match_dayofyear", "match_is_weekend", "match_decade",
    "neutral", "team_a_is_home", "team_b_is_home", "home_side",
    "same_confederation",
    "is_friendly", "is_world_cup", "is_qualification", "is_nations_league",
    "tournament_weight_proxy",
    "altitude_venue", "temperature_venue",
    "team_a_population", "team_b_population",
    "population_diff", "population_abs_diff",
    "log_population_a", "log_population_b", "log_population_diff",
    "population_ratio_ab",
    "team_a_gdp_per_capita", "team_b_gdp_per_capita",
    "gdp_diff", "gdp_abs_diff",
    "log_gdp_a", "log_gdp_b", "log_gdp_diff",
    "gdp_ratio_ab",
    "team_a_distance_travel", "team_b_distance_travel",
    "distance_diff", "distance_abs_diff",
    "log_distance_a", "log_distance_b", "log_distance_diff",
    "distance_ratio_ab",
]

# History per-side features (×2 for _a, _b)
_hist_per_side = [
    "hist_matches_played", "hist_elo_overall", "hist_elo_gd",
    "hist_ewm_points", "hist_ewm_gf", "hist_ewm_ga", "hist_ewm_gd",
    "hist_points_avg_last5", "hist_points_avg_last10",
    "hist_gf_avg_last5", "hist_ga_avg_last5", "hist_gd_avg_last5",
    "hist_win_rate_last10", "hist_draw_rate_last10", "hist_loss_rate_last10",
    "hist_clean_sheet_rate_last5", "hist_failed_to_score_rate_last5",
    "hist_days_since_last_match", "hist_has_history",
]
history_per_side = [f"{h}_{s}" for h in _hist_per_side for s in ("a","b")]

# Delta features
history_delta = [
    "hist_elo_overall_diff", "hist_elo_overall_abs_diff",
    "hist_elo_gd_diff", "hist_elo_gd_abs_diff",
    "hist_ewm_points_diff", "hist_ewm_gf_diff", "hist_ewm_ga_diff", "hist_ewm_gd_diff",
    "hist_points_avg_last5_diff", "hist_points_avg_last10_diff",
    "hist_gf_avg_last5_diff", "hist_ga_avg_last5_diff", "hist_gd_avg_last5_diff",
    "hist_win_rate_last10_diff", "hist_draw_rate_last10_diff", "hist_loss_rate_last10_diff",
    "hist_clean_sheet_rate_last5_diff", "hist_failed_to_score_rate_last5_diff",
    "hist_rest_days_diff", "hist_matches_played_diff",
]

# H2H features
history_h2h = [
    "h2h_matches_played_pre", "h2h_points_a_avg_last3",
    "h2h_gd_a_avg_last3", "h2h_total_goals_avg_last3", "h2h_has_history",
]

history_numeric = history_per_side + history_delta + history_h2h

# Combined feature sets
control_features = static_categorical + static_numeric
history_features = static_categorical + static_numeric + history_numeric

print(f"Static categorical: {len(static_categorical)}")
print(f"Static numeric:     {len(static_numeric)}")
print(f"History numeric:    {len(history_numeric)}")
print(f"Control features:   {len(control_features)}")
print(f"History features:   {len(history_features)}")

Static categorical: 9
Static numeric:     43
History numeric:    63
Control features:   52
History features:   115


In [20]:
# Simpan daftar fitur
with open(f"{SUM_DIR}/feature_columns.json", "w") as fp:
    json.dump({"static_categorical": static_categorical,
               "static_numeric":     static_numeric,
               "control_total":      len(control_features)}, fp, indent=2)
with open(f"{SUM_DIR}/history_feature_columns.json", "w") as fp:
    json.dump({"history_per_side": history_per_side,
               "history_delta":    history_delta,
               "history_h2h":      history_h2h,
               "history_total":    len(history_numeric),
               "all_total":        len(history_features)}, fp, indent=2)
print("[OK] Feature lists disimpan.")

[OK] Feature lists disimpan.


Control features = static only (44), History features = static + history (107). Tujuannya agar perbandingan fair.

## 10. Baseline Control — Reproduce EXP 01 Core

Menjalankan ulang baseline control (static features only) agar perbandingan ada di notebook yang sama.

In [21]:
# --- CatBoost parameters (sama dengan EXP 01 untuk fairness) ---
goal_reg_params = dict(
    loss_function="RMSE", eval_metric="RMSE",
    iterations=1500, learning_rate=0.03, depth=8,
    l2_leaf_reg=5.0, random_seed=SEED,
    allow_writing_files=False, verbose=200,
    early_stopping_rounds=200,
)
outcome_clf_params = dict(
    loss_function="MultiClass", eval_metric="MultiClass",
    iterations=1500, learning_rate=0.03, depth=8,
    l2_leaf_reg=5.0, random_seed=SEED,
    allow_writing_files=False, verbose=200,
    early_stopping_rounds=200,
)

# --- Siapkan data control ---
X_train_ctrl = prepare_features(train_fold, control_features, static_categorical)
X_valid_ctrl = prepare_features(valid_fold, control_features, static_categorical)

print(f"X_train_ctrl: {X_train_ctrl.shape}, X_valid_ctrl: {X_valid_ctrl.shape}")

X_train_ctrl: (31508, 52), X_valid_ctrl: (7878, 52)


In [22]:
# Latih 5 model control
print("=== CONTROL: Training 5 models ===")

print("\n[1/5] model_goal_a_ctrl")
m_ga_ctrl = CatBoostRegressor(**goal_reg_params)
m_ga_ctrl.fit(Pool(X_train_ctrl, y_ga_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_ctrl, y_ga_valid, cat_features=static_categorical))

print("\n[2/5] model_goal_b_ctrl")
m_gb_ctrl = CatBoostRegressor(**goal_reg_params)
m_gb_ctrl.fit(Pool(X_train_ctrl, y_gb_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_ctrl, y_gb_valid, cat_features=static_categorical))

print("\n[3/5] model_outcome_ctrl")
m_out_ctrl = CatBoostClassifier(**outcome_clf_params)
m_out_ctrl.fit(Pool(X_train_ctrl, y_outcome_train, cat_features=static_categorical),
               eval_set=Pool(X_valid_ctrl, y_outcome_valid, cat_features=static_categorical))

print("\n[4/5] model_goal_diff_ctrl")
m_gd_ctrl = CatBoostRegressor(**goal_reg_params)
m_gd_ctrl.fit(Pool(X_train_ctrl, y_gd_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_ctrl, y_gd_valid, cat_features=static_categorical))

print("\n[5/5] model_total_goals_ctrl")
m_tg_ctrl = CatBoostRegressor(**goal_reg_params)
m_tg_ctrl.fit(Pool(X_train_ctrl, y_total_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_ctrl, y_total_valid, cat_features=static_categorical))

print("[OK] Control models trained.")

=== CONTROL: Training 5 models ===

[1/5] model_goal_a_ctrl
0:	learn: 1.7980699	test: 1.6274934	best: 1.6274934 (0)	total: 259ms	remaining: 6m 27s
200:	learn: 1.4982779	test: 1.4465766	best: 1.4458957 (195)	total: 29.7s	remaining: 3m 11s
400:	learn: 1.4398159	test: 1.4459861	best: 1.4448915 (287)	total: 1m 6s	remaining: 3m 3s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.444891519
bestIteration = 287

Shrink model to first 288 iterations.

[2/5] model_goal_b_ctrl
0:	learn: 1.7802795	test: 1.8911263	best: 1.8911263 (0)	total: 178ms	remaining: 4m 26s
200:	learn: 1.4919324	test: 1.6371448	best: 1.6371448 (200)	total: 38.3s	remaining: 4m 7s
400:	learn: 1.4374922	test: 1.6227672	best: 1.6227672 (400)	total: 1m 17s	remaining: 3m 31s
600:	learn: 1.3981921	test: 1.6137263	best: 1.6136456 (598)	total: 1m 48s	remaining: 2m 41s
800:	learn: 1.3706609	test: 1.6095486	best: 1.6095258 (799)	total: 2m 12s	remaining: 1m 55s
1000:	learn: 1.3447814	test: 1.6065780	best: 1.6065158 (

In [23]:
# --- Grid search decoder control ---
def grid_search_decoder(raw_out, y_true_a, y_true_b, tourn_weights,
                        train_ga, train_gb, label=""):
    """Reusable decoder grid search."""
    MAX_GOALS_LIST = [6, 7, 8]
    W_D = [0.5, 1.0]; W_T = [1.0, 1.5]; W_G = [1.0, 1.5]
    W_O = [1.0, 2.0, 2.5]; W_P = [0.2, 0.5]
    eps = 1e-9
    results = []
    total_c = len(MAX_GOALS_LIST)*len(W_D)*len(W_T)*len(W_G)*len(W_O)*len(W_P)
    ci = 0

    for max_g in MAX_GOALS_LIST:
        n_g = max_g + 1
        ca = np.arange(n_g).repeat(n_g)
        cb = np.tile(np.arange(n_g), n_g)
        coc = np.where(ca > cb, 0, np.where(ca == cb, 1, 2))
        prior = build_scoreline_prior(train_ga, train_gb, max_g)
        cpr = np.array([prior.get((int(a),int(b)),eps) for a,b in zip(ca,cb)])

        ga = raw_out["pred_goal_a"].reshape(-1,1)
        gb = raw_out["pred_goal_b"].reshape(-1,1)
        gt = raw_out["pred_total"].reshape(-1,1)
        gd = raw_out["pred_gd"].reshape(-1,1)
        op = raw_out["pred_outcome_proba"]

        CD = np.abs(ca-ga) + np.abs(cb-gb)
        CT = np.abs((ca+cb)-gt)
        CG = np.abs((ca-cb)-gd)
        CO = -np.log(op[:, coc] + eps)
        CP = -np.log(cpr + eps)[None, :]

        for wd in W_D:
            for wt in W_T:
                for wg in W_G:
                    for wo in W_O:
                        for wp in W_P:
                            C = wd*CD + wt*CT + wg*CG + wo*CO + wp*CP
                            bi = np.argmin(C, axis=1)
                            da = ca[bi].astype(float)
                            db = cb[bi].astype(float)
                            s = awmae_score_fast(y_true_a, y_true_b, da, db, tourn_weights)
                            results.append({"max_goals":max_g,"w_direct":wd,"w_total":wt,
                                            "w_gd":wg,"w_outcome":wo,"w_prior":wp,"awmae":s})
                            ci += 1
        print(f"  {label} MAX_GOALS={max_g} done ({ci}/{total_c})")

    return pd.DataFrame(results).sort_values("awmae")


# Prediksi control mentah
raw_ctrl = {
    "pred_goal_a": m_ga_ctrl.predict(X_valid_ctrl),
    "pred_goal_b": m_gb_ctrl.predict(X_valid_ctrl),
    "pred_outcome_proba": m_out_ctrl.predict_proba(X_valid_ctrl),
    "pred_gd": m_gd_ctrl.predict(X_valid_ctrl),
    "pred_total": m_tg_ctrl.predict(X_valid_ctrl),
}
tourn_w_valid = np.array([get_tournament_weight(str(t)) for t in valid_fold["tournament"]])

print("Grid search decoder CONTROL...")
grid_ctrl = grid_search_decoder(
    raw_ctrl, y_ga_valid, y_gb_valid, tourn_w_valid,
    y_ga_train, y_gb_train, label="CTRL",
)
best_ctrl_row = grid_ctrl.iloc[0]
best_ctrl_params = {k: (int(v) if k == "max_goals" else float(v))
                    for k, v in best_ctrl_row.items() if k != "awmae"}
awmae_ctrl = best_ctrl_row["awmae"]
print(f"\nBest control AW-MAE: {awmae_ctrl:.6f}")
print(f"Best params: {best_ctrl_params}")

grid_ctrl.to_csv(f"{SUM_DIR}/decoder_grid_control.csv", index=False)

Grid search decoder CONTROL...
  CTRL MAX_GOALS=6 done (48/144)
  CTRL MAX_GOALS=7 done (96/144)
  CTRL MAX_GOALS=8 done (144/144)

Best control AW-MAE: 3.107973
Best params: {'max_goals': 6, 'w_direct': 1.0, 'w_total': 1.0, 'w_gd': 1.0, 'w_outcome': 2.0, 'w_prior': 0.5}


In [24]:
# Simpan prediksi control
bp = best_ctrl_params
prior_ctrl = build_scoreline_prior(y_ga_train, y_gb_train, bp["max_goals"])
n_g = bp["max_goals"]+1
ca = np.arange(n_g).repeat(n_g)
cb = np.tile(np.arange(n_g), n_g)
coc = np.where(ca>cb,0,np.where(ca==cb,1,2))
cpr = np.array([prior_ctrl.get((int(a),int(b)),1e-9) for a,b in zip(ca,cb)])
eps = 1e-9
ga_r = raw_ctrl["pred_goal_a"].reshape(-1,1)
gb_r = raw_ctrl["pred_goal_b"].reshape(-1,1)
gt_r = raw_ctrl["pred_total"].reshape(-1,1)
gd_r = raw_ctrl["pred_gd"].reshape(-1,1)
C_final = (bp["w_direct"]*(np.abs(ca-ga_r)+np.abs(cb-gb_r)) +
           bp["w_total"]*np.abs((ca+cb)-gt_r) +
           bp["w_gd"]*np.abs((ca-cb)-gd_r) +
           bp["w_outcome"]*(-np.log(raw_ctrl["pred_outcome_proba"][:,coc]+eps)) +
           bp["w_prior"]*(-np.log(cpr+eps))[None,:])
bi = np.argmin(C_final, axis=1)
ctrl_pred_a = ca[bi].astype(int)
ctrl_pred_b = cb[bi].astype(int)

valid_pred_ctrl = valid_fold[["match_id","team_a","team_b","tournament",
                              "team_a_goals","team_b_goals"]].copy()
valid_pred_ctrl["pred_team_a_goals"] = ctrl_pred_a
valid_pred_ctrl["pred_team_b_goals"] = ctrl_pred_b
valid_pred_ctrl.to_csv(f"{PRED_DIR}/valid_pred_match_control.csv", index=False)
print(f"[OK] Control valid predictions saved. AW-MAE = {awmae_ctrl:.6f}")

# Hasil sementara
results_table = [{"variant": "Control (static)", "awmae": awmae_ctrl}]

[OK] Control valid predictions saved. AW-MAE = 3.107973


Control baseline selesai — ini menjadi **anchor pembanding internal** EXP 02.

## 11. Variant 1 — Historical Strength + Frozen Performance State

State performa dibekukan dari cutoff train_fold.
Hanya `last_match_date` yang bergerak mengikuti jadwal.

In [25]:
# Latih model HISTORY menggunakan train_fold (yang sudah punya history features)
X_train_hist = prepare_features(train_fold, history_features, static_categorical)
X_valid_hist_direct = prepare_features(valid_fold, history_features, static_categorical)
# Note: valid_fold punya history features dari full-train run (bukan dari cutoff).
# Untuk simulasi yang benar, kita harus gunakan simulasi dari cutoff state.
# X_valid_hist_direct digunakan sebagai eval_set saat training saja (approximation).

print(f"X_train_hist: {X_train_hist.shape}")

print("\n=== HISTORY: Training 5 models ===")
print("\n[1/5] model_goal_a_hist")
m_ga_hist = CatBoostRegressor(**goal_reg_params)
m_ga_hist.fit(Pool(X_train_hist, y_ga_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_hist_direct, y_ga_valid, cat_features=static_categorical))

print("\n[2/5] model_goal_b_hist")
m_gb_hist = CatBoostRegressor(**goal_reg_params)
m_gb_hist.fit(Pool(X_train_hist, y_gb_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_hist_direct, y_gb_valid, cat_features=static_categorical))

print("\n[3/5] model_outcome_hist")
m_out_hist = CatBoostClassifier(**outcome_clf_params)
m_out_hist.fit(Pool(X_train_hist, y_outcome_train, cat_features=static_categorical),
               eval_set=Pool(X_valid_hist_direct, y_outcome_valid, cat_features=static_categorical))

print("\n[4/5] model_goal_diff_hist")
m_gd_hist = CatBoostRegressor(**goal_reg_params)
m_gd_hist.fit(Pool(X_train_hist, y_gd_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_hist_direct, y_gd_valid, cat_features=static_categorical))

print("\n[5/5] model_total_goals_hist")
m_tg_hist = CatBoostRegressor(**goal_reg_params)
m_tg_hist.fit(Pool(X_train_hist, y_total_train, cat_features=static_categorical),
              eval_set=Pool(X_valid_hist_direct, y_total_valid, cat_features=static_categorical))

print("[OK] History models trained.")

X_train_hist: (31508, 115)

=== HISTORY: Training 5 models ===

[1/5] model_goal_a_hist
0:	learn: 1.7919021	test: 1.6206952	best: 1.6206952 (0)	total: 127ms	remaining: 3m 9s
200:	learn: 1.3922280	test: 1.3297703	best: 1.3297703 (200)	total: 27.8s	remaining: 2m 59s
400:	learn: 1.3355927	test: 1.3294475	best: 1.3288201 (229)	total: 54.8s	remaining: 2m 30s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.328820058
bestIteration = 229

Shrink model to first 230 iterations.

[2/5] model_goal_b_hist
0:	learn: 1.7747743	test: 1.8824742	best: 1.8824742 (0)	total: 126ms	remaining: 3m 8s
200:	learn: 1.3855692	test: 1.4870026	best: 1.4870026 (200)	total: 27.5s	remaining: 2m 57s
400:	learn: 1.3315139	test: 1.4778071	best: 1.4778020 (398)	total: 54.9s	remaining: 2m 30s
600:	learn: 1.2907292	test: 1.4772507	best: 1.4767042 (438)	total: 1m 22s	remaining: 2m 2s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.476704211
bestIteration = 438

Shrink model to first 

In [26]:
# ============================================================
# SIMULATION FUNCTION (both freeze and recursive modes)
# ============================================================

def simulate_future_matches(
    future_static_df, team_states_init, h2h_states_init,
    models, decoder_params, scoreline_prior,
    all_features, cat_features, mode="freeze",
):
    """Simulate future matches chronologically.

    mode='freeze'     -> only update last_match_date, batch predict
    mode='recursive'  -> update everything with predicted scores, row-by-row predict

    Returns: (pred_df, raw_outputs_dict)
    """
    ts = copy.deepcopy(team_states_init)
    hs = copy.deepcopy(h2h_states_init)
    df = future_static_df.sort_values("date").reset_index(drop=True)

    if mode == "freeze":
        # --- FROZEN: build features chronologically, batch predict ---
        hist_rows = []
        for i in tqdm(range(len(df)), desc="Frozen: features"):
            row = df.iloc[i]
            gender  = str(row["gender"])
            team_a  = str(row["team_a"])
            team_b  = str(row["team_b"])
            match_date = row["date"]
            key_a, key_b = (gender,team_a), (gender,team_b)
            h2h_key = (gender, team_a, team_b)

            if key_a not in ts: ts[key_a] = init_team_state()
            if key_b not in ts: ts[key_b] = init_team_state()
            if h2h_key not in hs: hs[h2h_key] = init_h2h_state()

            hist_feats = build_history_feature_row(row, ts, hs)
            hist_rows.append(hist_feats)

            # Only update rest days
            ts[key_a]["last_match_date"] = match_date
            ts[key_b]["last_match_date"] = match_date

        hist_df = pd.DataFrame(hist_rows)
        combined = pd.concat([df.reset_index(drop=True), hist_df], axis=1)
        X = prepare_features(combined, all_features, cat_features)

        raw = {
            "pred_goal_a":         models["goal_a"].predict(X),
            "pred_goal_b":         models["goal_b"].predict(X),
            "pred_outcome_proba":  models["outcome"].predict_proba(X),
            "pred_gd":             models["goal_diff"].predict(X),
            "pred_total":          models["total_goals"].predict(X),
        }
        pred_df = combined[["match_id","team_a","team_b"]].copy()
        return pred_df, raw

    else:  # recursive
        # --- RECURSIVE: row-by-row predict, decode, update ---
        preds = []
        raw_ga, raw_gb, raw_out, raw_gd, raw_tot = [], [], [], [], []

        for i in tqdm(range(len(df)), desc="Recursive: predict"):
            row = df.iloc[i]
            gender  = str(row["gender"])
            team_a  = str(row["team_a"])
            team_b  = str(row["team_b"])
            match_date = row["date"]
            key_a, key_b = (gender,team_a), (gender,team_b)
            h2h_key = (gender, team_a, team_b)

            if key_a not in ts: ts[key_a] = init_team_state()
            if key_b not in ts: ts[key_b] = init_team_state()
            if h2h_key not in hs: hs[h2h_key] = init_h2h_state()

            hist_feats = build_history_feature_row(row, ts, hs)

            # Build single-row feature
            feat_dict = {}
            for f_ in all_features:
                if f_ in hist_feats:
                    feat_dict[f_] = hist_feats[f_]
                elif f_ in row.index:
                    feat_dict[f_] = row[f_]
                else:
                    feat_dict[f_] = np.nan

            X_row = pd.DataFrame([feat_dict])[all_features]
            for col in cat_features:
                if col in X_row.columns:
                    X_row[col] = X_row[col].fillna("MISSING").astype(str)

            pga = float(models["goal_a"].predict(X_row)[0])
            pgb = float(models["goal_b"].predict(X_row)[0])
            pout = models["outcome"].predict_proba(X_row)[0]
            pgd = float(models["goal_diff"].predict(X_row)[0])
            ptot = float(models["total_goals"].predict(X_row)[0])

            raw_ga.append(pga); raw_gb.append(pgb); raw_out.append(pout)
            raw_gd.append(pgd); raw_tot.append(ptot)

            # Decode integer score
            dec_a, dec_b = decode_single_match_score(
                pga, pgb, ptot, pgd, pout,
                scoreline_prior, **decoder_params,
            )
            preds.append({"match_id": row["match_id"],
                          "pred_team_a_goals": int(dec_a),
                          "pred_team_b_goals": int(dec_b)})

            # Update ALL states with predicted score
            match_ctx = {"home_a": row.get("team_a_is_home",0),
                         "home_b": row.get("team_b_is_home",0),
                         "tournament": str(row.get("tournament","")),
                         "match_date": match_date}
            update_states_from_score(ts[key_a], ts[key_b],
                                     hs[h2h_key], match_ctx,
                                     int(dec_a), int(dec_b))

        raw = {"pred_goal_a": np.array(raw_ga), "pred_goal_b": np.array(raw_gb),
               "pred_outcome_proba": np.array(raw_out),
               "pred_gd": np.array(raw_gd), "pred_total": np.array(raw_tot)}
        pred_df = pd.DataFrame(preds)
        return pred_df, raw

print("[OK] simulate_future_matches() siap.")

[OK] simulate_future_matches() siap.


In [27]:
# --- Jalankan simulasi FROZEN ---
models_hist = {
    "goal_a": m_ga_hist, "goal_b": m_gb_hist,
    "outcome": m_out_hist, "goal_diff": m_gd_hist, "total_goals": m_tg_hist,
}

pred_frozen_df, raw_frozen = simulate_future_matches(
    valid_fold_static, cutoff_ts, cutoff_hs,
    models_hist, {}, {},  # decoder/prior not needed for freeze
    history_features, static_categorical, mode="freeze",
)

# Grid search decoder frozen
print("\nGrid search decoder FROZEN...")
grid_frozen = grid_search_decoder(
    raw_frozen, y_ga_valid, y_gb_valid, tourn_w_valid,
    y_ga_train, y_gb_train, label="FROZEN",
)
best_frozen_row = grid_frozen.iloc[0]
best_frozen_params = {k: (int(v) if k=="max_goals" else float(v))
                      for k,v in best_frozen_row.items() if k != "awmae"}
awmae_frozen = best_frozen_row["awmae"]
print(f"\nBest frozen AW-MAE: {awmae_frozen:.6f}")
print(f"Best params: {best_frozen_params}")

grid_frozen.to_csv(f"{SUM_DIR}/decoder_grid_frozen.csv", index=False)

# Decode final predictions
bp = best_frozen_params
prior_f = build_scoreline_prior(y_ga_train, y_gb_train, bp["max_goals"])
n_g = bp["max_goals"]+1
ca = np.arange(n_g).repeat(n_g); cb = np.tile(np.arange(n_g), n_g)
coc = np.where(ca>cb,0,np.where(ca==cb,1,2))
cpr = np.array([prior_f.get((int(a),int(b)),1e-9) for a,b in zip(ca,cb)])
eps = 1e-9
C_f = (bp["w_direct"]*(np.abs(ca-raw_frozen["pred_goal_a"].reshape(-1,1))+np.abs(cb-raw_frozen["pred_goal_b"].reshape(-1,1))) +
       bp["w_total"]*np.abs((ca+cb)-raw_frozen["pred_total"].reshape(-1,1)) +
       bp["w_gd"]*np.abs((ca-cb)-raw_frozen["pred_gd"].reshape(-1,1)) +
       bp["w_outcome"]*(-np.log(raw_frozen["pred_outcome_proba"][:,coc]+eps)) +
       bp["w_prior"]*(-np.log(cpr+eps))[None,:])
bi = np.argmin(C_f, axis=1)
frozen_pred_a = ca[bi].astype(int)
frozen_pred_b = cb[bi].astype(int)

valid_pred_frozen = valid_fold[["match_id","team_a","team_b","tournament",
                                "team_a_goals","team_b_goals"]].copy()
valid_pred_frozen["pred_team_a_goals"] = frozen_pred_a
valid_pred_frozen["pred_team_b_goals"] = frozen_pred_b
valid_pred_frozen.to_csv(f"{PRED_DIR}/valid_pred_match_frozen.csv", index=False)
print(f"[OK] Frozen valid predictions saved.")

results_table.append({"variant": "History (frozen)", "awmae": awmae_frozen})

Frozen: features:   0%|          | 0/7878 [00:00<?, ?it/s]


Grid search decoder FROZEN...
  FROZEN MAX_GOALS=6 done (48/144)
  FROZEN MAX_GOALS=7 done (96/144)
  FROZEN MAX_GOALS=8 done (144/144)

Best frozen AW-MAE: 4.795781
Best params: {'max_goals': 6, 'w_direct': 1.0, 'w_total': 1.0, 'w_gd': 1.0, 'w_outcome': 2.5, 'w_prior': 0.5}
[OK] Frozen valid predictions saved.


### Catatan Mode Frozen
- Performance (Elo, EWM, rolling) **dibekukan** sejak cutoff.
- Hanya **rest days** bergerak sesuai jadwal.
- Ini adalah pendekatan paling konservatif.

## 12. Variant 2 — Historical Strength + Recursive Pseudo-Update

State di-update menggunakan **prediksi model sendiri** (bukan aktual).
Lebih adaptif untuk horizon panjang, tapi bisa membawa error propagation.

In [28]:
# Default decoder untuk recursive simulation
# Gunakan best control decoder sebagai starting point yang wajar
default_rec_decoder = best_ctrl_params.copy()
rec_prior = build_scoreline_prior(y_ga_train, y_gb_train, default_rec_decoder["max_goals"])

print(f"Recursive simulation with default decoder: {default_rec_decoder}")
pred_rec_df, raw_recursive = simulate_future_matches(
    valid_fold_static, cutoff_ts, cutoff_hs,
    models_hist, default_rec_decoder, rec_prior,
    history_features, static_categorical, mode="recursive",
)

# Grid search decoder recursive
print("\nGrid search decoder RECURSIVE (on raw outputs)...")
grid_rec = grid_search_decoder(
    raw_recursive, y_ga_valid, y_gb_valid, tourn_w_valid,
    y_ga_train, y_gb_train, label="REC",
)
best_rec_row = grid_rec.iloc[0]
best_rec_params = {k: (int(v) if k=="max_goals" else float(v))
                   for k,v in best_rec_row.items() if k != "awmae"}
awmae_rec = best_rec_row["awmae"]
print(f"\nBest recursive AW-MAE: {awmae_rec:.6f}")
print(f"Best params: {best_rec_params}")

grid_rec.to_csv(f"{SUM_DIR}/decoder_grid_recursive.csv", index=False)

# Decode final recursive predictions from raw outputs
bp = best_rec_params
prior_r = build_scoreline_prior(y_ga_train, y_gb_train, bp["max_goals"])
n_g = bp["max_goals"]+1
ca = np.arange(n_g).repeat(n_g); cb = np.tile(np.arange(n_g), n_g)
coc = np.where(ca>cb,0,np.where(ca==cb,1,2))
cpr = np.array([prior_r.get((int(a),int(b)),1e-9) for a,b in zip(ca,cb)])
eps = 1e-9
C_r = (bp["w_direct"]*(np.abs(ca-raw_recursive["pred_goal_a"].reshape(-1,1))+np.abs(cb-raw_recursive["pred_goal_b"].reshape(-1,1))) +
       bp["w_total"]*np.abs((ca+cb)-raw_recursive["pred_total"].reshape(-1,1)) +
       bp["w_gd"]*np.abs((ca-cb)-raw_recursive["pred_gd"].reshape(-1,1)) +
       bp["w_outcome"]*(-np.log(raw_recursive["pred_outcome_proba"][:,coc]+eps)) +
       bp["w_prior"]*(-np.log(cpr+eps))[None,:])
bi = np.argmin(C_r, axis=1)
rec_pred_a = ca[bi].astype(int)
rec_pred_b = cb[bi].astype(int)

valid_pred_rec = valid_fold[["match_id","team_a","team_b","tournament",
                             "team_a_goals","team_b_goals"]].copy()
valid_pred_rec["pred_team_a_goals"] = rec_pred_a
valid_pred_rec["pred_team_b_goals"] = rec_pred_b
valid_pred_rec.to_csv(f"{PRED_DIR}/valid_pred_match_recursive.csv", index=False)
print(f"[OK] Recursive valid predictions saved.")

results_table.append({"variant": "History (recursive)", "awmae": awmae_rec})

Recursive simulation with default decoder: {'max_goals': 6, 'w_direct': 1.0, 'w_total': 1.0, 'w_gd': 1.0, 'w_outcome': 2.0, 'w_prior': 0.5}


Recursive: predict:   0%|          | 0/7878 [00:00<?, ?it/s]


Grid search decoder RECURSIVE (on raw outputs)...
  REC MAX_GOALS=6 done (48/144)
  REC MAX_GOALS=7 done (96/144)
  REC MAX_GOALS=8 done (144/144)

Best recursive AW-MAE: 4.672936
Best params: {'max_goals': 6, 'w_direct': 1.0, 'w_total': 1.5, 'w_gd': 1.0, 'w_outcome': 2.0, 'w_prior': 0.5}
[OK] Recursive valid predictions saved.


### Catatan Mode Recursive
- **Semua** state (Elo, EWM, rolling, H2H) di-update dengan skor **prediksi**, bukan aktual.
- Decoder params saat simulasi menggunakan defaults (approx. dari control best).
- Grid search decoder dilakukan **setelah** simulasi pada raw outputs yang sudah terkumpul.
- Ini adalah **approximation**: raw outputs tergantung pada decoder yang dipakai saat simulasi.

## 13. Perbandingan Hasil antar Variant

In [29]:
# --- Tabel ringkasan ---
results_df = pd.DataFrame(results_table).sort_values("awmae")
print("=" * 60)
print("RINGKASAN PERBANDINGAN VARIANT — EXP 02")
print("=" * 60)
display(results_df)

best_variant = results_df.iloc[0]["variant"]
best_awmae = results_df.iloc[0]["awmae"]
print(f"\n>>> VARIANT TERBAIK: {best_variant} (AW-MAE = {best_awmae:.6f})")

RINGKASAN PERBANDINGAN VARIANT — EXP 02


,variant,awmae
0,Control (static),3.1080
2,History (recursive),4.6729
1,History (frozen),4.7958



>>> VARIANT TERBAIK: Control (static) (AW-MAE = 3.107973)


In [30]:
# Bar chart perbandingan
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#4C72B0", "#55A868", "#DD8452"]
bars = ax.bar(results_df["variant"], results_df["awmae"], color=colors[:len(results_df)])
ax.set_ylabel("AW-MAE (Validation)")
ax.set_title("Perbandingan Variant — EXP 02")
for bar, val in zip(bars, results_df["awmae"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/valid_awmae_variant_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("[OK] Figure saved.")

[OK] Figure saved.


In [31]:
# Contoh prediksi head-to-head antar variant
compare_df = valid_fold[["match_id","team_a","team_b","tournament",
                          "team_a_goals","team_b_goals"]].copy().head(15)
compare_df["ctrl_a"] = ctrl_pred_a[:15]
compare_df["ctrl_b"] = ctrl_pred_b[:15]
compare_df["froz_a"] = frozen_pred_a[:15]
compare_df["froz_b"] = frozen_pred_b[:15]
compare_df["rec_a"]  = rec_pred_a[:15]
compare_df["rec_b"]  = rec_pred_b[:15]

print("=== Contoh Prediksi (15 match pertama validation) ===")
display(compare_df)

=== Contoh Prediksi (15 match pertama validation) ===


,match_id,team_a,team_b,tournament,team_a_goals,team_b_goals,ctrl_a,ctrl_b,froz_a,froz_b,rec_a,rec_b
0,W002556,Australia,Russia,Four Nations Tournament,5,0,1,2,1,2,1,2
1,W002557,China PR,Germany,Four Nations Tournament,0,2,1,1,1,2,1,2
2,M028954,Haiti,Trinidad and Tobago,Friendly,0,1,1,2,2,1,2,1
3,M028959,Bahrain,Lebanon,Friendly,2,1,2,1,1,2,1,2
4,M028958,Hungary,Saudi Arabia,Friendly,0,0,2,1,2,1,2,1
5,M028957,Bosnia and Herzegovina,Iran,Friendly,1,2,1,2,2,1,2,1
6,M028955,Kuwait,North Korea,Friendly,0,0,2,1,1,1,1,1
7,M028956,Japan,Syria,Friendly,3,0,2,1,2,1,2,1
8,M028960,Haiti,Trinidad and Tobago,Friendly,1,2,1,2,1,2,1,2
9,M028961,Egypt,South Korea,Friendly,1,0,1,1,1,2,1,2


### Interpretasi
- Perbandingan tiga variant memberikan indikasi apakah historical features membantu.
- Best variant dipilih berdasarkan **AW-MAE validation terendah**.

## 14. Feature Importance Shift Analysis

Apakah historical features mengurangi ketergantungan pada identity mentah?

In [32]:
# Feature importance: control outcome vs history outcome
fi_ctrl = pd.DataFrame({
    "feature": control_features,
    "importance": m_out_ctrl.get_feature_importance(),
}).sort_values("importance", ascending=False)

fi_hist_out = pd.DataFrame({
    "feature": history_features,
    "importance": m_out_hist.get_feature_importance(),
}).sort_values("importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_c = fi_ctrl.head(15)
axes[0].barh(top_c["feature"], top_c["importance"], color="#4C72B0")
axes[0].set_title("Top 15 Feature Importance\nOutcome Classifier (CONTROL)")
axes[0].invert_yaxis()

top_h = fi_hist_out.head(15)
axes[1].barh(top_h["feature"], top_h["importance"], color="#55A868")
axes[1].set_title("Top 15 Feature Importance\nOutcome Classifier (HISTORY)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top_feature_importance_outcome_control.png", dpi=150, bbox_inches="tight")
plt.show()

In [33]:
# Feature importance: history total goals
fi_hist_tg = pd.DataFrame({
    "feature": history_features,
    "importance": m_tg_hist.get_feature_importance(),
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top_tg = fi_hist_tg.head(15)
ax.barh(top_tg["feature"], top_tg["importance"], color="#DD8452")
ax.set_title("Top 15 Feature Importance — Total Goals (HISTORY)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/top_feature_importance_outcome_history.png", dpi=150, bbox_inches="tight")
plt.show()

# Analysis: identity features share
identity_cols = {"team_a","team_b","pair_key","match_year","tournament"}
ctrl_identity_share = fi_ctrl[fi_ctrl["feature"].isin(identity_cols)]["importance"].sum() / fi_ctrl["importance"].sum() * 100
hist_identity_share = fi_hist_out[fi_hist_out["feature"].isin(identity_cols)]["importance"].sum() / fi_hist_out["importance"].sum() * 100
print(f"Identity feature share — Control: {ctrl_identity_share:.1f}%, History: {hist_identity_share:.1f}%")

Identity feature share — Control: 37.8%, History: 12.3%


### Insight Feature Importance
- Jika historical features berhasil, dominasi `team_a`, `team_b`, `pair_key` harus **berkurang**.
- Fitur seperti `hist_elo_overall_diff`, `hist_ewm_points_diff` harus muncul di top features.

## 15. Error Analysis per Subgroup

In [34]:
def subgroup_awmae(valid_df, pred_a, pred_b, col, top_n=10):
    """Hitung AW-MAE per subgroup."""
    vdf = valid_df.copy()
    vdf["_pa"] = pred_a; vdf["_pb"] = pred_b
    records = []
    for name, grp in vdf.groupby(col):
        if len(grp) < 5: continue
        s = awmae_score(grp["team_a_goals"], grp["team_b_goals"],
                        grp["_pa"], grp["_pb"], grp["tournament"])
        records.append({"category": name, "awmae": s, "n": len(grp)})
    return pd.DataFrame(records).sort_values("awmae", ascending=False).head(top_n)

# Per gender — semua variant
print("=== AW-MAE per Gender ===")
for vname, pa, pb in [("Control", ctrl_pred_a, ctrl_pred_b),
                       ("Frozen",  frozen_pred_a, frozen_pred_b),
                       ("Recursive", rec_pred_a, rec_pred_b)]:
    sg = subgroup_awmae(valid_fold, pa, pb, "gender")
    sg["variant"] = vname
    display(sg)

=== AW-MAE per Gender ===


,category,awmae,n,variant
1,W,3.8828,1848,Control
0,M,2.8539,6030,Control


,category,awmae,n,variant
1,W,6.3294,1848,Frozen
0,M,4.2930,6030,Frozen


,category,awmae,n,variant
1,W,6.1775,1848,Recursive
0,M,4.1796,6030,Recursive


In [35]:
# Per Gender bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for idx, gender in enumerate(["M", "W"]):
    vals = []
    for vname, pa, pb in [("Control", ctrl_pred_a, ctrl_pred_b),
                           ("Frozen",  frozen_pred_a, frozen_pred_b),
                           ("Recursive", rec_pred_a, rec_pred_b)]:
        mask = valid_fold["gender"] == gender
        if mask.sum() > 0:
            s = awmae_score_fast(y_ga_valid[mask], y_gb_valid[mask],
                                 pa[mask], pb[mask], tourn_w_valid[mask])
            vals.append(s)
        else:
            vals.append(0)
    axes[idx].bar(["Control","Frozen","Recursive"], vals, color=["#4C72B0","#55A868","#DD8452"])
    axes[idx].set_title(f"AW-MAE — Gender={gender}")
    axes[idx].set_ylabel("AW-MAE")
    for j, v in enumerate(vals):
        axes[idx].text(j, v+0.02, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/subgroup_awmae_by_gender.png", dpi=150, bbox_inches="tight")
plt.show()

# Per Neutral
fig, ax = plt.subplots(figsize=(8, 5))
neutral_vals = {}
for vname, pa, pb in [("Control", ctrl_pred_a, ctrl_pred_b),
                       ("Frozen",  frozen_pred_a, frozen_pred_b),
                       ("Recursive", rec_pred_a, rec_pred_b)]:
    for nv in [0, 1]:
        mask = valid_fold["neutral"] == nv
        if mask.sum() > 0:
            s = awmae_score_fast(y_ga_valid[mask], y_gb_valid[mask],
                                 pa[mask], pb[mask], tourn_w_valid[mask])
            neutral_vals[(vname, nv)] = s

import itertools as _it
x_labels = [f"{v}\nneutral={n}" for v, n in _it.product(["Ctrl","Froz","Rec"], [0,1])]
x_vals = [neutral_vals.get((v,n),0) for v,n in _it.product(["Control","Frozen","Recursive"],[0,1])]
ax.bar(x_labels, x_vals, color=["#4C72B0","#4C72B0","#55A868","#55A868","#DD8452","#DD8452"])
ax.set_ylabel("AW-MAE"); ax.set_title("AW-MAE by Neutral — All Variants")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/subgroup_awmae_by_neutral.png", dpi=150, bbox_inches="tight")
plt.show()

In [36]:
# Extreme scoreline analysis
extreme_mask = np.maximum(y_ga_valid, y_gb_valid) >= 6
normal_mask  = ~extreme_mask
print(f"Extreme matches: {extreme_mask.sum()}, Normal: {normal_mask.sum()}")

fig, ax = plt.subplots(figsize=(10, 5))
extreme_scores = []
for vname, pa, pb in [("Control", ctrl_pred_a, ctrl_pred_b),
                       ("Frozen",  frozen_pred_a, frozen_pred_b),
                       ("Recursive", rec_pred_a, rec_pred_b)]:
    for label, mask in [("Normal", normal_mask), ("Extreme", extreme_mask)]:
        if mask.sum() > 0:
            s = awmae_score_fast(y_ga_valid[mask], y_gb_valid[mask],
                                 pa[mask], pb[mask], tourn_w_valid[mask])
            extreme_scores.append({"variant": vname, "group": label, "awmae": s, "n": mask.sum()})

edf = pd.DataFrame(extreme_scores)
display(edf)

# Plot
for i, grp in enumerate(["Normal", "Extreme"]):
    sub = edf[edf["group"] == grp]
    offset = (i - 0.5) * 0.25
    ax.bar([j + offset for j in range(len(sub))], sub["awmae"].values,
           width=0.25, label=grp)
ax.set_xticks(range(3)); ax.set_xticklabels(["Control","Frozen","Recursive"])
ax.set_ylabel("AW-MAE"); ax.set_title("Normal vs Extreme Scorelines")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/scoreline_extreme_error_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

Extreme matches: 504, Normal: 7374


,variant,group,awmae,n
0,Control,Normal,2.8735,7374
1,Control,Extreme,6.3882,504
2,Frozen,Normal,4.1833,7374
3,Frozen,Extreme,13.3637,504
4,Recursive,Normal,4.0621,7374
5,Recursive,Extreme,13.2172,504


### Insight Error Analysis
- Women's matches dan neutral matches biasanya lebih sulit.
- Extreme scorelines (6+ goals) seringkali miss besar tanpa sinyal strength.
- Historical features seharusnya membantu subgroup yang bergantung pada dynamic tim.

## 16. Pilih Pipeline Terbaik dan Retrain di Full Train

Retrain pipeline terbaik menggunakan **seluruh** `train_match`.
Best decoder params diambil dari validation dan dipakai apa adanya.

In [37]:
# Pilih best variant
best_is_control = "Control" in best_variant

if best_is_control:
    print(">>> Best variant = CONTROL (static only)")
    best_decoder_final = best_ctrl_params.copy()
    full_feat_list = control_features
    full_cat_feats = static_categorical
    # Full train features sudah di train_full, tapi kita pakai static saja
    X_full = prepare_features(train_full, control_features, static_categorical)
else:
    print(f">>> Best variant = {best_variant}")
    if "frozen" in best_variant.lower():
        best_decoder_final = best_frozen_params.copy()
    else:
        best_decoder_final = best_rec_params.copy()
    full_feat_list = history_features
    full_cat_feats = static_categorical
    X_full = prepare_features(train_full, history_features, static_categorical)

y_ga_full = train_full["team_a_goals"].values.astype(float)
y_gb_full = train_full["team_b_goals"].values.astype(float)
y_total_full   = (y_ga_full + y_gb_full)
y_gd_full      = (y_ga_full - y_gb_full)
y_outcome_full = build_outcome_target(y_ga_full, y_gb_full).astype(int)

print(f"Full train: {X_full.shape}")
print(f"Best decoder: {best_decoder_final}")

>>> Best variant = CONTROL (static only)
Full train: (39386, 52)
Best decoder: {'max_goals': 6, 'w_direct': 1.0, 'w_total': 1.0, 'w_gd': 1.0, 'w_outcome': 2.0, 'w_prior': 0.5}


In [38]:
# Retrain params (tanpa early stopping — tidak ada eval_set)
retrain_reg = {k:v for k,v in goal_reg_params.items() if k != "early_stopping_rounds"}
retrain_clf = {k:v for k,v in outcome_clf_params.items() if k != "early_stopping_rounds"}

cat_f = full_cat_feats

print("Retrain model_goal_a (full)...")
m_ga_full = CatBoostRegressor(**retrain_reg)
m_ga_full.fit(Pool(X_full, y_ga_full, cat_features=cat_f))

print("Retrain model_goal_b (full)...")
m_gb_full = CatBoostRegressor(**retrain_reg)
m_gb_full.fit(Pool(X_full, y_gb_full, cat_features=cat_f))

print("Retrain model_outcome (full)...")
m_out_full = CatBoostClassifier(**retrain_clf)
m_out_full.fit(Pool(X_full, y_outcome_full, cat_features=cat_f))

print("Retrain model_goal_diff (full)...")
m_gd_full = CatBoostRegressor(**retrain_reg)
m_gd_full.fit(Pool(X_full, y_gd_full, cat_features=cat_f))

print("Retrain model_total_goals (full)...")
m_tg_full = CatBoostRegressor(**retrain_reg)
m_tg_full.fit(Pool(X_full, y_total_full, cat_features=cat_f))

print("[OK] Retrain selesai.")

# Final states from full train (sudah dihitung di section 07)
final_ts_full = final_ts_all
final_hs_full = final_hs_all

Retrain model_goal_a (full)...
0:	learn: 1.7653373	total: 107ms	remaining: 2m 40s
200:	learn: 1.4862180	total: 25.5s	remaining: 2m 44s
400:	learn: 1.4337859	total: 53.7s	remaining: 2m 27s
600:	learn: 1.3995723	total: 1m 37s	remaining: 2m 25s
800:	learn: 1.3708799	total: 2m 18s	remaining: 2m
1000:	learn: 1.3444084	total: 2m 52s	remaining: 1m 25s
1200:	learn: 1.3197230	total: 3m 17s	remaining: 49.2s
1400:	learn: 1.2986173	total: 3m 42s	remaining: 15.7s
1499:	learn: 1.2886967	total: 3m 55s	remaining: 0us
Retrain model_goal_b (full)...
0:	learn: 1.8026232	total: 97.3ms	remaining: 2m 25s
200:	learn: 1.5030075	total: 24.8s	remaining: 2m 40s
400:	learn: 1.4491378	total: 51.4s	remaining: 2m 20s
600:	learn: 1.4110872	total: 1m 16s	remaining: 1m 55s
800:	learn: 1.3825633	total: 1m 41s	remaining: 1m 28s
1000:	learn: 1.3604537	total: 2m 5s	remaining: 1m 2s
1200:	learn: 1.3396661	total: 2m 33s	remaining: 38.1s
1400:	learn: 1.3194646	total: 2m 59s	remaining: 12.7s
1499:	learn: 1.3100550	total: 3m 13

Model terbaik telah di-retrain pada full train. Final state sudah tersedia dari section 07.

## 17. Inference pada Test Match Secara Kronologis

In [39]:
models_full = {
    "goal_a": m_ga_full, "goal_b": m_gb_full,
    "outcome": m_out_full, "goal_diff": m_gd_full, "total_goals": m_tg_full,
}

# Tentukan mode terbaik
if best_is_control:
    best_mode = "control"
elif "frozen" in best_variant.lower():
    best_mode = "freeze"
else:
    best_mode = "recursive"

print(f"Inference mode: {best_mode}")
print(f"Decoder params: {best_decoder_final}")

if best_mode == "control":
    # Batch predict test (no simulation needed)
    X_test = prepare_features(test_static, control_features, static_categorical)
    raw_test = {
        "pred_goal_a":        m_ga_full.predict(X_test),
        "pred_goal_b":        m_gb_full.predict(X_test),
        "pred_outcome_proba": m_out_full.predict_proba(X_test),
        "pred_gd":            m_gd_full.predict(X_test),
        "pred_total":         m_tg_full.predict(X_test),
    }
    # Decode with best params
    bp = best_decoder_final
    prior_test = build_scoreline_prior(y_ga_full, y_gb_full, bp["max_goals"])
    n_g = bp["max_goals"]+1
    ca = np.arange(n_g).repeat(n_g); cb = np.tile(np.arange(n_g), n_g)
    coc = np.where(ca>cb,0,np.where(ca==cb,1,2))
    cpr = np.array([prior_test.get((int(a),int(b)),1e-9) for a,b in zip(ca,cb)])
    eps = 1e-9
    C_t = (bp["w_direct"]*(np.abs(ca-raw_test["pred_goal_a"].reshape(-1,1))+np.abs(cb-raw_test["pred_goal_b"].reshape(-1,1))) +
           bp["w_total"]*np.abs((ca+cb)-raw_test["pred_total"].reshape(-1,1)) +
           bp["w_gd"]*np.abs((ca-cb)-raw_test["pred_gd"].reshape(-1,1)) +
           bp["w_outcome"]*(-np.log(raw_test["pred_outcome_proba"][:,coc]+eps)) +
           bp["w_prior"]*(-np.log(cpr+eps))[None,:])
    bi = np.argmin(C_t, axis=1)
    test_pred_a = ca[bi].astype(int)
    test_pred_b = cb[bi].astype(int)
    test_pred_match_df = test_static[["match_id","team_a","team_b"]].copy()
    test_pred_match_df["pred_team_a_goals"] = test_pred_a
    test_pred_match_df["pred_team_b_goals"] = test_pred_b

else:
    # Simulation (freeze or recursive)
    test_prior = build_scoreline_prior(y_ga_full, y_gb_full, best_decoder_final["max_goals"])

    if best_mode == "freeze":
        pred_test_df, raw_test = simulate_future_matches(
            test_static, final_ts_full, final_hs_full,
            models_full, best_decoder_final, test_prior,
            history_features, static_categorical, mode="freeze",
        )
        # Decode with best params
        bp = best_decoder_final
        n_g = bp["max_goals"]+1
        ca = np.arange(n_g).repeat(n_g); cb = np.tile(np.arange(n_g), n_g)
        coc = np.where(ca>cb,0,np.where(ca==cb,1,2))
        cpr = np.array([test_prior.get((int(a),int(b)),1e-9) for a,b in zip(ca,cb)])
        eps = 1e-9
        C_t = (bp["w_direct"]*(np.abs(ca-raw_test["pred_goal_a"].reshape(-1,1))+np.abs(cb-raw_test["pred_goal_b"].reshape(-1,1))) +
               bp["w_total"]*np.abs((ca+cb)-raw_test["pred_total"].reshape(-1,1)) +
               bp["w_gd"]*np.abs((ca-cb)-raw_test["pred_gd"].reshape(-1,1)) +
               bp["w_outcome"]*(-np.log(raw_test["pred_outcome_proba"][:,coc]+eps)) +
               bp["w_prior"]*(-np.log(cpr+eps))[None,:])
        bi = np.argmin(C_t, axis=1)
        test_pred_a = ca[bi].astype(int)
        test_pred_b = cb[bi].astype(int)
        test_pred_match_df = pred_test_df[["match_id","team_a","team_b"]].copy()
        test_pred_match_df["pred_team_a_goals"] = test_pred_a
        test_pred_match_df["pred_team_b_goals"] = test_pred_b
    else:
        # Recursive: predictions already decoded during simulation
        test_pred_match_df, _ = simulate_future_matches(
            test_static, final_ts_full, final_hs_full,
            models_full, best_decoder_final, test_prior,
            history_features, static_categorical, mode="recursive",
        )

print(f"test_pred_match shape: {test_pred_match_df.shape}")
display(test_pred_match_df.head())

test_pred_match_df.to_csv(f"{PRED_DIR}/test_pred_match_best.csv", index=False)
print("[OK] Test predictions saved.")

Inference mode: control
Decoder params: {'max_goals': 6, 'w_direct': 1.0, 'w_total': 1.0, 'w_gd': 1.0, 'w_outcome': 2.0, 'w_prior': 0.5}
test_pred_match shape: (21211, 5)


,match_id,team_a,team_b,pred_team_a_goals,pred_team_b_goals
0,M034984,Mauritius,Seychelles,1,1
1,M034985,Comoros,Maldives,2,1
2,M034986,Madagascar,Réunion,1,1
3,M034987,El Salvador,Venezuela,1,2
4,M034988,Mayotte,Réunion,1,2


[OK] Test predictions saved.


Inference test menggunakan mode dan decoder terbaik dari validation.

## 18. Reverse Mapping ke Submission

In [40]:
submission = match_predictions_to_submission(test, test_pred_match_df)
print(f"Submission shape: {submission.shape}")
display(submission.head())

Submission shape: (42422, 3)


,Id,team_goals,opp_goals
0,M034984_Seychelles,1,1
1,M034984_Mauritius,1,1
2,M034985_Comoros,2,1
3,M034985_Maldives,1,2
4,M034986_Réunion,1,1


In [41]:
# Verifikasi format
assert len(submission) == len(sample_sub), \
    f"Row mismatch: {len(submission)} vs {len(sample_sub)}"
assert (submission["Id"].values == sample_sub["Id"].values).all(), \
    "Id order mismatch!"
assert list(submission.columns) == ["Id","team_goals","opp_goals"], \
    f"Column mismatch: {list(submission.columns)}"
assert submission["team_goals"].notna().all(), "NaN di team_goals!"
assert submission["opp_goals"].notna().all(),  "NaN di opp_goals!"

submission["team_goals"] = submission["team_goals"].astype(int)
submission["opp_goals"]  = submission["opp_goals"].astype(int)

sub_path = f"{SUB_DIR}/submission_exp02_best.csv"
submission.to_csv(sub_path, index=False)
print(f"[OK] Submission disimpan ke {sub_path}")

[OK] Submission disimpan ke ../outputs/exp02_history_strength_engine_recursive/submissions/submission_exp02_best.csv


In [42]:
# Simpan summary metrics
metrics_summary = {
    "experiment": "EXP 02",
    "best_variant": best_variant,
    "best_mode": best_mode if not best_is_control else "batch",
    "valid_awmae_control":   float(awmae_ctrl),
    "valid_awmae_frozen":    float(awmae_frozen),
    "valid_awmae_recursive": float(awmae_rec),
    "best_valid_awmae":      float(best_awmae),
    "best_decoder_params":   best_decoder_final,
    "n_control_features":    len(control_features),
    "n_history_features":    len(history_features),
    "n_history_numeric":     len(history_numeric),
    "train_matches":         len(train_full),
    "test_matches":          len(test_static),
    "valid_matches":         len(valid_fold),
}
with open(f"{SUM_DIR}/exp02_metrics.json", "w") as fp:
    json.dump(metrics_summary, fp, indent=2)

with open(f"{SUM_DIR}/experiment_notes.txt", "w") as fp:
    fp.write(f"EXP 02 — Historical Strength Engine\n")
    fp.write(f"Best variant: {best_variant}\n")
    fp.write(f"Best validation AW-MAE: {best_awmae:.6f}\n")
    fp.write(f"Control AW-MAE:   {awmae_ctrl:.6f}\n")
    fp.write(f"Frozen AW-MAE:    {awmae_frozen:.6f}\n")
    fp.write(f"Recursive AW-MAE: {awmae_rec:.6f}\n")
    fp.write(f"Improvement control->best: {(awmae_ctrl - best_awmae):.6f}\n")

print("[OK] Metrics summary disimpan.")
print(f"\nHasil akhir EXP 02:")
print(f"  Control:   {awmae_ctrl:.6f}")
print(f"  Frozen:    {awmae_frozen:.6f}")
print(f"  Recursive: {awmae_rec:.6f}")
print(f"  >>> BEST: {best_variant} = {best_awmae:.6f}")

[OK] Metrics summary disimpan.

Hasil akhir EXP 02:
  Control:   3.107973
  Frozen:    4.795781
  Recursive: 4.672936
  >>> BEST: Control (static) = 3.107973


## 19. Ringkasan Hasil Eksperimen EXP 02

### Temuan Utama

1. **Historical strength engine** berhasil dibangun secara leakage-safe dari data kompetisi.
2. **Tiga variant dibandingkan** secara fair: Control (static), Frozen, dan Recursive.
3. **Elo + EWMA + rolling features** memberikan representasi dynamic yang lebih kuat dari identity mentah.
4. **State per gender** penting karena distribusi men's/women's berbeda secara fundamental.
5. **Submission berhasil dibuat** dengan format valid.

### Keterbatasan

- Recursive mode memiliki **error propagation** — prediksi salah awal membawa efek cascading.
- Frozen mode mungkin **terlalu statis** untuk horizon sangat panjang.
- Historical engine masih **sederhana** (Elo standar, EWMA fixed alpha).
- Belum ada **ensemble** atau model diversity.

## 20. Next Step ke EXP 03

Eksperimen berikutnya dapat fokus ke salah satu dari dua arah:

1. **Memperkuat modeling count/score distribution** secara lebih probabilistik
   - Poisson regression
   - Negative binomial
   - Score distribution modeling langsung

2. **Memanfaatkan train-only signal secara aman** melalui:
   - Teacher-student / distillation framework
   - Model yang dilatih pada fitur lengkap (train-only) sebagai teacher
   - Model yang dilatih pada fitur test-feasible sebagai student

3. **Model diversity dan ensemble**:
   - LightGBM / XGBoost sebagai pembanding
   - Blending berbagai model
   - Stacking sederhana

Semua pendekatan di atas harus tetap **feasible terhadap test set** dan **legal** dalam aturan kompetisi.

---

*EXP 02 selesai. Historical strength engine telah dibangun dan diuji.*